<a href="https://colab.research.google.com/github/er1cbrown/BiostatsPython/blob/main/Diabetic_Retinopathy_EDA_APTOS_%2B_EyePACS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
tanlikesmath_diabetic_retinopathy_resized_path = kagglehub.dataset_download('tanlikesmath/diabetic-retinopathy-resized')

print('Data source import complete.')


# Diabetic Retinopathy Grading: EDA (APTOS 2019 + EyePACS 2015)

Exploratory analysis of the two main public fundus-photo datasets graded on the ICDR **0–4** scale:

| Grade | Meaning |
|---|---|
| 0 | No DR |
| 1 | Mild NPDR |
| 2 | Moderate NPDR |
| 3 | Severe NPDR |
| 4 | Proliferative DR |

**Inputs** (attached to this notebook):
- `aptos2019-blindness-detection` (competition, ~3.7k labeled PNGs)
- `tanlikesmath/diabetic-retinopathy-resized` (EyePACS 2015 train set, ~35k JPEGs, resized)

**What this covers:** label distributions and imbalance, image geometry, the black-border crop, brightness/color differences between datasets, image-quality checks (blur, darkness), patient-level structure in EyePACS (left/right eyes), near-duplicate detection, and a leakage-safe train/val split saved to CSV.

In [ ]:
import os, glob, random, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
SEED = 42; random.seed(SEED); np.random.seed(SEED)

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/kaggle/input"))
SAMPLE_PER_SOURCE = int(os.environ.get("SAMPLE_PER_SOURCE", 1500))  # images per dataset for pixel-level stats
GRADE_NAMES = {0: "No DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}
PALETTE = {"APTOS": "#2a78b5", "EyePACS": "#d9822b"}
print("Data root:", DATA_ROOT)
for p in sorted(DATA_ROOT.iterdir()): print(" -", p.name)

## 1. Locate files and load labels
Paths are discovered with `glob`, so the notebook still works if the dataset folder names differ slightly.

In [ ]:
def find_one(pattern):
    hits = sorted(DATA_ROOT.glob(pattern))
    assert hits, f"Nothing matches {pattern} under {DATA_ROOT}"
    return hits[0]

# APTOS
aptos_csv = find_one("**/aptos*/train.csv")
aptos_img_dir = aptos_csv.parent / "train_images"
aptos = pd.read_csv(aptos_csv).rename(columns={"id_code": "image_id", "diagnosis": "grade"})
aptos["path"] = aptos.image_id.map(lambda i: str(aptos_img_dir / f"{i}.png"))
aptos["source"] = "APTOS"
aptos["patient_id"] = aptos.image_id          # APTOS has no patient info
aptos["eye"] = np.nan

# EyePACS: prefer the uncropped label file
label_files = sorted(DATA_ROOT.glob("**/trainLabels*.csv"), key=lambda p: ("cropped" in p.name, len(str(p))))
eyepacs_csv = label_files[0]
eye_imgs = {Path(p).stem: p for p in glob.glob(str(DATA_ROOT / "**/resized_train/**/*.jpeg"), recursive=True)}
if not eye_imgs:
    eye_imgs = {Path(p).stem: p for p in glob.glob(str(eyepacs_csv.parent / "**/*.jpeg"), recursive=True)}
eyepacs = pd.read_csv(eyepacs_csv).rename(columns={"image": "image_id", "level": "grade"})
eyepacs["path"] = eyepacs.image_id.map(eye_imgs)
missing = eyepacs.path.isna().sum()
eyepacs = eyepacs.dropna(subset=["path"])
eyepacs["source"] = "EyePACS"
eyepacs["patient_id"] = eyepacs.image_id.str.rsplit("_", n=1).str[0]
eyepacs["eye"] = eyepacs.image_id.str.rsplit("_", n=1).str[1]

df = pd.concat([aptos, eyepacs], ignore_index=True)[["image_id", "patient_id", "eye", "grade", "source", "path"]]
df["grade"] = df.grade.astype(int)
print(f"APTOS:   {len(aptos):>6,} images  ({aptos_csv})")
print(f"EyePACS: {len(eyepacs):>6,} images  ({eyepacs_csv}); {missing} label rows had no image file")
df.head()

## 2. Label distribution and class imbalance

In [ ]:
counts = df.groupby(["source", "grade"]).size().unstack(0).fillna(0).astype(int)
pct = counts / counts.sum() * 100
summary = pd.concat({"count": counts, "%": pct.round(1)}, axis=1)
summary.index = [f"{g} – {GRADE_NAMES[g]}" for g in summary.index]
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (src, sub) in zip(axes, counts.items()):
    bars = ax.bar(sub.index, sub.values, color=PALETTE[src])
    ax.bar_label(bars, labels=[f"{v:,}\n{p:.1f}%" for v, p in zip(sub.values, pct[src])], fontsize=9)
    ax.set(title=f"{src} (n={sub.sum():,})", xlabel="Grade", ylabel="Images", xticks=range(5))
    ax.set_ylim(0, sub.max() * 1.2)
plt.suptitle("Grade distribution per dataset", y=1.02, fontsize=14); plt.tight_layout(); plt.show()

ax = pct.plot(kind="bar", figsize=(9, 4), color=[PALETTE[c] for c in pct.columns], rot=0)
ax.set(title="Relative grade distribution (%)", xlabel="Grade", ylabel="% of dataset"); plt.show()

for src in counts:
    c = counts[src]
    print(f"{src}: majority/minority ratio = {c.max() / c.min():.1f}x | referable DR (grade >= 2) = {c[2:].sum() / c.sum():.1%}")

**Things to note**
- Both datasets are dominated by grade 0, and EyePACS much more so (~73%). Plain accuracy is meaningless here: always predicting 0 scores ~73% on EyePACS. Use **quadratic weighted kappa (QWK)**.
- Grades 3 and 4 are rare, so you will need class weighting, oversampling, or an ordinal/regression loss.
- The grade mix differs between datasets, so a model trained on one will be miscalibrated on the other. Keep track of where each image comes from in any combined training.

## 3. Image geometry and file properties (sampled)
Reading headers and pixels for every image is slow, so pixel-level statistics use a stratified sample per dataset (`SAMPLE_PER_SOURCE`).

In [ ]:
def stratified_sample(d, n):
    out = []
    for g, grp in d.groupby("grade"):
        k = max(1, int(round(n * len(grp) / len(d))))
        out.append(grp.sample(min(k, len(grp)), random_state=SEED))
    return pd.concat(out)

sample = pd.concat([stratified_sample(d, SAMPLE_PER_SOURCE) for _, d in df.groupby("source")], ignore_index=True)

def fundus_mask(img, thresh=10):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return gray > thresh

def image_stats(path):
    img = cv2.imread(path)
    if img is None:
        return {}
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    m = fundus_mask(img)
    fg = img[m] if m.any() else img.reshape(-1, 3)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (512, int(512 * h / w)))
    return dict(width=w, height=h, aspect=w / h, file_kb=os.path.getsize(path) / 1024,
                fg_frac=m.mean(), mean_r=fg[:, 0].mean(), mean_g=fg[:, 1].mean(), mean_b=fg[:, 2].mean(),
                brightness=fg.mean(), contrast=fg.std(),
                blur=cv2.Laplacian(small, cv2.CV_64F)[cv2.resize(m.astype(np.uint8), small.shape[::-1]) > 0].var())

with ThreadPoolExecutor(8) as ex:
    stats = pd.DataFrame(list(ex.map(image_stats, sample.path)))
sample = pd.concat([sample.reset_index(drop=True), stats], axis=1).dropna(subset=["width"])
sample.groupby("source")[["width", "height", "aspect", "file_kb", "fg_frac"]].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for src, sub in sample.groupby("source"):
    axes[0].scatter(sub.width, sub.height, s=8, alpha=.4, label=src, color=PALETTE[src])
    sns.kdeplot(sub.aspect, ax=axes[1], label=src, color=PALETTE[src], fill=True, warn_singular=False)
    sns.kdeplot(sub.fg_frac, ax=axes[2], label=src, color=PALETTE[src], fill=True, warn_singular=False)
axes[0].set(title="Resolution (width × height)", xlabel="width px", ylabel="height px")
axes[1].set(title="Aspect ratio (w/h)"); axes[2].set(title="Fraction of image that is retina (not black border)")
for a in axes: a.legend()
plt.tight_layout(); plt.show()
print(sample.groupby("source")[["width", "height"]].agg(lambda s: s.value_counts().head(3).to_dict()))

APTOS resolutions vary a lot because the images come from several clinics and cameras. Many are wide, with large black side borders, and some have the top and bottom of the retina cut off. The resized EyePACS images are more uniform. In both datasets a large share of each image is black background, which is pixels and compute spent on nothing. That is the reason for the crop in the next section.

## 4. Sample images per grade

In [ ]:
def load_rgb(p, size=None):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (size, size)) if size else img

N_PER = 5
for src, sub in df.groupby("source"):
    fig, axes = plt.subplots(5, N_PER, figsize=(N_PER * 2.6, 5 * 2.6))
    for g in range(5):
        pool = sub[sub.grade == g]
        picks = pool.sample(min(N_PER, len(pool)), random_state=SEED) if len(pool) else pool
        for j in range(N_PER):
            ax = axes[g, j]; ax.axis("off")
            if j < len(picks):
                ax.imshow(load_rgb(picks.iloc[j].path, 300))
            if j == 0:
                ax.set_title(f"Grade {g}: {GRADE_NAMES[g]}", loc="left", fontsize=10)
    plt.suptitle(f"{src}: random samples per grade", fontsize=14, y=1.0); plt.tight_layout(); plt.show()

## 5. Preprocessing: crop the black border

Threshold the grayscale image, keep the bounding box of the retina, then pad to a square. Optionally apply **Ben Graham's** local-contrast normalization (subtract a Gaussian blur), from the 2015 Kaggle winning solution, which evens out lighting differences between cameras.

In [ ]:
def crop_black_border(img, tol=10, pad_square=True):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if not mask.any():
        return img
    rows, cols = np.where(mask.any(1))[0], np.where(mask.any(0))[0]
    img = img[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
    if pad_square:
        h, w = img.shape[:2]; s = max(h, w)
        top, left = (s - h) // 2, (s - w) // 2
        img = cv2.copyMakeBorder(img, top, s - h - top, left, s - w - left, cv2.BORDER_CONSTANT, value=0)
    return img

def ben_graham(img, size=512, sigma=10):
    img = cv2.resize(img, (size, size))
    out = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigma), -4, 128)
    circ = np.zeros_like(img); cv2.circle(circ, (size // 2, size // 2), int(size * 0.48), (1, 1, 1), -1)
    return out * circ + 128 * (1 - circ)

demo = pd.concat([sample[sample.source == s].nsmallest(2, "fg_frac") for s in PALETTE] +
                 [sample[sample.source == s].sample(1, random_state=1) for s in PALETTE])
fig, axes = plt.subplots(len(demo), 3, figsize=(11, 3.4 * len(demo)))
for i, (_, r) in enumerate(demo.iterrows()):
    raw = load_rgb(r.path); crop = crop_black_border(raw)
    for ax, im, t in zip(axes[i], [raw, crop, ben_graham(crop)],
                         [f"{r.source} raw {raw.shape[1]}×{raw.shape[0]}", f"cropped {crop.shape[1]}×{crop.shape[0]}", "cropped + Ben Graham"]):
        ax.imshow(im.astype(np.uint8)); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Brightness and color differences between datasets
Different cameras and clinics give different color profiles. If brightness or color correlates with grade, a model can learn the shortcut instead of the lesions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.boxplot(data=sample, x="grade", y="brightness", hue="source", palette=PALETTE, ax=axes[0], fliersize=1)
sns.boxplot(data=sample, x="grade", y="contrast", hue="source", palette=PALETTE, ax=axes[1], fliersize=1)
ch = sample.melt(id_vars="source", value_vars=["mean_r", "mean_g", "mean_b"], var_name="channel")
sns.violinplot(data=ch, x="channel", y="value", hue="source", palette=PALETTE, split=True, ax=axes[2], inner="quart")
axes[0].set_title("Retina brightness by grade"); axes[1].set_title("Retina contrast (std) by grade"); axes[2].set_title("Mean channel intensity")
plt.tight_layout(); plt.show()
display(sample.groupby("source")[["mean_r", "mean_g", "mean_b", "brightness", "contrast"]].mean().round(1))

In [ ]:
# Does file size or resolution predict grade? A strong correlation is a shortcut-learning warning sign.
corr = (sample.groupby("source")
        .apply(lambda d: d[["width", "file_kb", "brightness", "contrast", "blur", "fg_frac"]].corrwith(d.grade, method="spearman"))
        .round(3))
display(corr.style.background_gradient(cmap="RdBu_r", vmin=-.4, vmax=.4).set_caption("Spearman correlation with grade"))

In APTOS, image width and file size are known to correlate with grade, because different clinics contributed different shares of severe cases. A CNN can partly "grade" from capture conditions. Crop and resize every image to the same square size so that resolution can't leak, and check results across datasets.

## 7. Image quality: blur and underexposure
Low variance of the Laplacian means a blurry image. Very low brightness means an underexposed one. EyePACS is known to contain many images too poor to grade.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for src, sub in sample.groupby("source"):
    sns.histplot(np.log10(sub.blur + 1), ax=axes[0], label=src, color=PALETTE[src], stat="density", bins=50, alpha=.5)
    sns.histplot(sub.brightness, ax=axes[1], label=src, color=PALETTE[src], stat="density", bins=50, alpha=.5)
axes[0].set(title="Sharpness: log10 Laplacian variance (low = blurry)"); axes[1].set(title="Retina brightness (low = underexposed)")
for a in axes: a.legend()
plt.tight_layout(); plt.show()

blur_thr = sample.blur.quantile(.05); dark_thr = sample.brightness.quantile(.05)
sample["low_quality"] = (sample.blur < blur_thr) | (sample.brightness < dark_thr)
print(f"Flagged (bottom 5% sharpness or brightness): blur < {blur_thr:.1f}, brightness < {dark_thr:.1f}")
display(sample.groupby(["source", "grade"]).low_quality.mean().unstack(0).mul(100).round(1).rename(columns=lambda c: f"{c} % flagged"))

worst = pd.concat([sample.nsmallest(4, "blur"), sample.nsmallest(4, "brightness")])
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, r) in zip(axes.ravel(), worst.iterrows()):
    ax.imshow(load_rgb(r.path, 300)); ax.axis("off")
    ax.set_title(f"{r.source} g{r.grade} blur={r.blur:.0f} bright={r.brightness:.0f}", fontsize=8)
plt.suptitle("Lowest-quality images in the sample (top: blurriest, bottom: darkest)"); plt.tight_layout(); plt.show()

## 8. EyePACS patient structure: left and right eyes
Each EyePACS patient has two images. The two eyes are strongly correlated, so a random image-level split puts the same patient in both train and validation and inflates scores. **Split by patient.**

In [ ]:
pairs = eyepacs.pivot_table(index="patient_id", columns="eye", values="grade", aggfunc="first").dropna().astype(int)
print(f"Patients: {eyepacs.patient_id.nunique():,} | with both eyes: {len(pairs):,}")
agree = (pairs.left == pairs.right).mean(); within1 = ((pairs.left - pairs.right).abs() <= 1).mean()
print(f"Left/right exact grade agreement: {agree:.1%} | within ±1 grade: {within1:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm = pd.crosstab(pairs.left, pairs.right)
sns.heatmap(np.log10(cm + 1), annot=cm, fmt=",", cmap="Blues", ax=axes[0], cbar=False)
axes[0].set(title="Left vs right eye grade (counts, log color)", xlabel="Right eye", ylabel="Left eye")
pairs.assign(diff=pairs.right - pairs.left)["diff"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color=PALETTE["EyePACS"], rot=0)
axes[1].set(title="Right minus left grade", xlabel="Difference", ylabel="Patients")
plt.tight_layout(); plt.show()

Because the two eyes agree so often, a model that sees both eyes of a patient can use the other eye as extra input. That is a common trick in strong EyePACS solutions, and also why patient-level splitting is essential.

## 9. Near-duplicate check (APTOS)
APTOS is known to contain duplicate and near-duplicate images, some with **conflicting grades**. A 64-bit difference hash (dHash) on the cropped image finds them.

In [ ]:
def dhash(path, size=8):
    img = crop_black_border(load_rgb(path), pad_square=True)
    g = cv2.resize(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (size + 1, size))
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

with ThreadPoolExecutor(8) as ex:
    aptos["dhash"] = list(ex.map(dhash, aptos.path))

H = np.array(aptos.dhash.tolist(), dtype=np.uint64)
bits = np.unpackbits(H.view(np.uint8).reshape(-1, 8), axis=1)
MAX_DIST = 3
dup_pairs = []
for i in range(len(bits)):
    d = (bits[i + 1:] != bits[i]).sum(1)
    for j in np.where(d <= MAX_DIST)[0]:
        dup_pairs.append((i, i + 1 + j, int(d[j])))
dups = pd.DataFrame(dup_pairs, columns=["i", "j", "hamming"])
if len(dups):
    dups["grade_i"] = aptos.grade.values[dups.i]; dups["grade_j"] = aptos.grade.values[dups.j]
    dups["conflict"] = dups.grade_i != dups.grade_j
print(f"Near-duplicate pairs (Hamming <= {MAX_DIST}): {len(dups)}; with conflicting grades: {int(dups.conflict.sum()) if len(dups) else 0}")

show = dups.sort_values("conflict", ascending=False).head(4) if len(dups) else dups
if len(show):
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 3 * len(show)), squeeze=False)
    for k, (_, r) in enumerate(show.iterrows()):
        for ax, idx in zip(axes[k], [r.i, r.j]):
            ax.imshow(load_rgb(aptos.path.iloc[idx], 256)); ax.axis("off")
            ax.set_title(f"{aptos.image_id.iloc[idx][:10]}… grade {aptos.grade.iloc[idx]}", fontsize=8)
    plt.suptitle("Example near-duplicates (conflicting grades first)"); plt.tight_layout(); plt.show()

Before training: drop near-duplicates whose grades conflict, merge the agreeing ones, and keep every duplicate group on the **same side** of any split.

## 10. Leakage-safe train/validation split
Stratified by grade and grouped by patient (EyePACS) or duplicate cluster (APTOS). The result is saved to `/kaggle/working/dr_splits.csv` for the modelling notebook.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Group APTOS duplicates together using union-find over the duplicate pairs
parent = list(range(len(aptos)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for i, j in zip(dups.get("i", []), dups.get("j", [])):
    parent[find(i)] = find(j)
aptos["group"] = ["aptos_" + str(find(i)) for i in range(len(aptos))]
conflict_groups = aptos.groupby("group").grade.nunique().loc[lambda s: s > 1].index
aptos["drop_conflict"] = aptos.group.isin(conflict_groups)

eyepacs["group"] = "eyepacs_" + eyepacs.patient_id
eyepacs["drop_conflict"] = False
cols = ["image_id", "patient_id", "eye", "grade", "source", "path", "group", "drop_conflict"]
split = pd.concat([aptos[cols], eyepacs[cols]], ignore_index=True)

split["fold"] = -1
for src, idx in split.groupby("source").groups.items():
    sub = split.loc[idx]
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for f, (_, va) in enumerate(sgkf.split(sub, sub.grade, sub.group)):
        split.loc[sub.index[va], "fold"] = f

out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
split.to_csv(out / "dr_splits.csv", index=False)
print(f"Saved {len(split):,} rows to {out / 'dr_splits.csv'}; {int(split.drop_conflict.sum())} APTOS images flagged as conflicting duplicates")
display(pd.crosstab([split.source, split.fold], split.grade, normalize="index").mul(100).round(1))

## 11. Summary and next steps

**Findings**
1. **Heavy imbalance.** Grade 0 makes up about half of APTOS and about three-quarters of EyePACS, and grades 3 and 4 are rare. Use **QWK** as the metric and an ordinal or regression loss with tuned rounding thresholds.
2. **The datasets differ.** Resolution, field of view, color and grade mix all differ between APTOS and EyePACS. Normalize inputs (crop, then square resize, optionally Ben Graham) and always report results on each dataset separately.
3. **Black borders waste pixels.** Cropping recovers a large share of the useful resolution, especially in wide APTOS images.
4. **Shortcut risk.** Capture properties such as image size and file size correlate with grade in APTOS. Resize everything to the same size before the model sees it.
5. **Quality.** A noticeable tail of blurry and dark images exists, mainly in EyePACS. Consider filtering them, down-weighting them, or adding a quality-classification side task.
6. **Leakage.** Both eyes of an EyePACS patient, and APTOS near-duplicates, must stay on the same side of a split. That is handled in `dr_splits.csv`.

**Suggested modelling baseline**
- Preprocess: crop the border, resize to 384–512 px, light augmentation (flips, rotation, brightness and contrast).
- Backbone: EfficientNet-B3/B4 or ConvNeXt-T (ImageNet weights), or **RETFound** for a retina-specific starting point.
- Pretrain on EyePACS, fine-tune on APTOS, evaluate on the APTOS validation folds plus an external set such as Messidor-2.